In [ ]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

In [ ]:
!pip install -q faster-whisper pyannote.audio "numpy==2.2.6"

In [ ]:
!pip install faster-whisper "av<19" --quiet

In [ ]:
!apt-get -y install ffmpeg

In [ ]:
!pip install -q fastapi uvicorn python-multipart nest-asyncio pyngrok

In [ ]:
!pip install noisereduce soundfile

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install -q supabase

In [ ]:
import os, glob, ctypes
import nvidia.cublas.lib, nvidia.cudnn.lib

for pkg in (nvidia.cublas.lib, nvidia.cudnn.lib):
    lib_dir = list(pkg.__path__)[0]
    for so in sorted(glob.glob(os.path.join(lib_dir, "*.so*"))):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

In [ ]:
from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = "float16" if device == "cuda" else "int8"

whisper_model = WhisperModel("medium", device = device, compute_type = dtype)
print("Whisper Model loaded.")

In [ ]:
from pyannote.audio import Pipeline as DiarizationPipeline
from google.colab import userdata
import torch

hf_token = userdata.get("HF_TOKEN")
diarization_pipeline = DiarizationPipeline.from_pretrained("pyannote/speaker-diarization-3.1", token = hf_token).to(torch.device(device))

print("Diarization pipeline loaded")

In [ ]:
import subprocess
import os

def normalize_audio(input_path: str) -> str:
    """Convert uploaded audio to a clean 16kHz mono WAV to avoid
    pyannote sample-count mismatches with compressed/VBR audio."""
    output_path = os.path.splitext(input_path)[0] + "_normalized.wav"
    subprocess.run(
        ["ffmpeg", "-y", "-i", input_path, "-ar", "16000", "-ac", "1",
         "-sample_fmt", "s16", output_path],
        check=True,
        capture_output=True
    )
    return output_path

def diarize_audio(audio_path):
  audio_path = normalize_audio(audio_path)
  output = diarization_pipeline(audio_path)

  speaker_segments = []

  if hasattr(output, "speaker_diarization"):
    for turn, speaker in output.speaker_diarization:
      speaker_segments.append((turn.start, turn.end, speaker))
  else:
    for turn, _, speaker in output.itertracks(yield_label=True):
      speaker_segments.append((turn.start, turn.end, speaker))

  return speaker_segments

def assign_speakers_to_transcript(whisper_segments, speaker_segments, low_confidence_threshold = 0.75):
  labelled_segments = []
  for seg in whisper_segments:
    seg_mid = (seg.start + seg.end)/2
    best_speaker = 0
    best_overlap = 0

    for spk_start, spk_end, speaker in speaker_segments:
      overlap = min(spk_end, seg.end) - max(spk_start, seg.start)
      if overlap > best_overlap:
        best_overlap = overlap
        best_speaker = speaker

    low_conf = []
    if seg.words:
      low_conf = [w.word.strip() for w in seg.words if w.probability < low_confidence_threshold]

    labelled_segments.append({
        "start": round(seg.start, 2),
        "end": round(seg.end, 2),
        "speaker": best_speaker,
        "text": seg.text.strip(),
        "low_conf": low_conf,
    })

  return labelled_segments

def format_speaker_transcription(labelled_segments):
  lines = []
  current_speaker = None
  current_text = []

  for seg in labelled_segments:
    if seg["speaker"] != current_speaker:
      if current_text:
        lines.append(f"{current_speaker}: {' '.join(current_text)}")

      current_speaker = seg["speaker"]
      current_text = [seg["text"]]

    else:
      current_text.append(seg["text"])

  if current_text :
    lines.append(f"{current_speaker}: {' '.join(current_text)}")

  return "\n".join(lines)



In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

gemini_api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

print("Gemini client ready")

In [ ]:
import time
import json

SUMMARY_PROMPTS = {
    "narrative": """You are an expert summarizer of narrative/story content.

Summarize the given transcript in a single coherent summary.

Instructions:
- Capture the key events, characters, and ideas in chronological order.
- Remove filler, repetition, and unnecessary description.
- Preserve important names, dates, places, and technical terms.
- Write naturally in your own words - do not copy passages from the transcript.
- Produce approximately 400-600 words.

Transcript:
{text}""",

    "meeting": """You are an expert meeting analyst.

Summarize the given meeting transcript, structured as follows:

**Overview:** 2-3 sentence summary of the meeting's purpose.
**Key Discussion Points:** bullet list of main topics discussed.
**Decisions Made:** bullet list of any decisions reached.
**Action Items:** bullet list in format "- [Owner, if known]: Task"
**Open Questions / Follow-ups:** anything left unresolved.

Base this only on the transcript. If speaker labels are present, attribute key points/decisions to specific speakers where possible.

Transcript:
{text}""",

    "conference": """You are an expert at summarizing conference talks and presentations.

Summarize the given transcript, structured as follows:

**Topic & Speaker's Main Argument:** what is this talk fundamentally about.
**Key Points:** bullet list of the main supporting points or findings.
**Notable Quotes or Claims:** any particularly important or specific claims made (paraphrased, not verbatim).
**Conclusion / Takeaway:** what the speaker wants the audience to remember or do.

Transcript:
{text}""",

    "speech": """You are an expert at analyzing persuasive speeches.

Summarize the given transcript, structured as follows:

**Central Message:** the core argument or call to action.
**Key Rhetorical Points:** the main points used to build the argument, in order.
**Tone & Intent:** the emotional tone and apparent goal of the speech (e.g., to inspire, to persuade, to inform).
**Conclusion:** how the speech resolves or what it calls the audience to do.

Transcript:
{text}"""
}

def summarize_transcript(text, purpose = "narrative", max_retries = 3):
    if purpose not in SUMMARY_PROMPTS:
        print(f"Unknown purpose '{purpose}', defaulting to 'narrative'")
        purpose = "narrative"

    prompt = SUMMARY_PROMPTS[purpose].format(text = text)
    models_to_try = ["gemini-flash-latest", "gemini-3.6-flash"]

    last_error = None

    for model in models_to_try:
      for attempt in range(max_retries):
          try:
              response = client.models.generate_content(
                  model = model,
                  contents = prompt,
                  config = types.GenerateContentConfig(
                      thinking_config = types.ThinkingConfig(thinking_level = "low")
                  )
              )
              finish_reason = response.candidates[0].finish_reason
              if finish_reason != "STOP":
                print(f"Warning: response finished with reason '{finish_reason}', may be incomplete")
              return response.text.strip()
          except Exception as e:
              last_error = e
              print(f"API call failed (attempt {attempt + 1}/{max_retries}): {e}")

              if attempt < max_retries - 1:
                time.sleep(min(60, 5 * (2 ** attempt)))
      print(f"All attempts for {model} failed, trying next model if available...")

    raise last_error if last_error else RuntimeError("Summarization failed for unknown reasons")

FORMAT_PROMPTS = {
    "meeting": """Based on this meeting transcript and summary, write a follow-up email.
Return ONLY valid JSON, no markdown fences, no commentary, in this exact shape:
{{"subject": "...", "body": "..."}}
The body should include a brief recap and a clearly formatted action items list (use "- " bullets and \\n for line breaks).

SUMMARY:
{summary}

TRANSCRIPT:
{transcript}""",

        "conference": """Based on this talk's transcript and summary, extract shareable content.
Return ONLY valid JSON, no markdown fences, no commentary, in this exact shape:
{{"recap": "a 2-3 sentence recap suitable for a social post", "quotes": [{{"quote": "...", "context": "one short phrase on what it's about"}}]}}
Pick 3 to 5 quotes that meet ALL of these criteria:
- Under 25 words each.
- Grammatically clean as a standalone sentence — skip run-on or rambling lines even if the underlying idea is good.
- Self-contained: understandable without hearing the rest of the talk.
- Actually said in the transcript, word-for-word. Do not invent or paraphrase quotes.
If the transcript has informal, run-on, or grammatically rough speech throughout, it is better to return fewer high-quality quotes (even just 1 or 2) than to force 3-5 weak ones.

SUMMARY:
{summary}

TRANSCRIPT:
{transcript}""",

    "speech": None,

    "narrative": """Based on this story/narrative transcript, format it as a clean reading document.
Return ONLY valid JSON, no markdown fences, no commentary, in this exact shape:
{{"pages": [{{"heading": "...", "text": "..."}}]}}
Break it into 3 to 6 natural sections with short descriptive headings. Lightly clean up filler words but preserve the storyteller's actual wording and voice — do not summarize or shorten the content.

TRANSCRIPT:
{transcript}"""
}

FORMAT_PROMPTS["speech"] = FORMAT_PROMPTS["conference"]

def generate_formatted_output(purpose, text, summary, max_retries = 3):
  if purpose not in FORMAT_PROMPTS:
    purpose = "meeting"

  prompt = FORMAT_PROMPTS[purpose].format(transcript = text[:15000], summary = summary)
  models_to_try = ["gemini-flash-latest", "gemini-3.6-flash"]

  last_error = None

  for model in models_to_try:
    for attempt in range(max_retries):
      try:
        response = client.models.generate_content(
            model = model,
            contents = prompt,
            config = types.GenerateContentConfig(
                thinking_config = types.ThinkingConfig(thinking_level = "low")
            )
        )
        raw = response.text.strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

        json.loads(raw)

        return raw

      except json.JSONDecodeError as e:
        last_error = e
        print(f"Formatted output wasn't valid JSON (attempt {attempt + 1}/{max_retries}): {e}")

      except Exception as e:
        last_error = e
        print(f"API call failed (attempt {attempt + 1}/{max_retries}): {e}")

      if attempt < max_retries - 1:
        time.sleep(min(60, 5 * (2 ** attempt)))

    print(f"All attempts for {model} failed, trying next model if available...")

  print(f"Formatted output generation failed entirely: {last_error}")

  return json.dumps({"error": "Formatting failed to generate"})

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from sentence_transformers import SentenceTransformer, util

retriever_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Retriever Model loaded.")

In [ ]:
import re
from sentence_transformers import SentenceTransformer, CrossEncoder, util

retriever_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
reranker = CrossEncoder("BAAI/bge-reranker-base")

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def build_chunks(segments, purpose, speaker_names=None, max_words=150):
    """Group consecutive segments into chunks, remembering start/end times."""
    speaker_names = speaker_names or {}
    chunks, cur, words = [], [], 0

    def make_chunk(segs):
        if purpose == "meeting":
            text = "\n".join(
                f"{speaker_names.get(s['speaker'], s['speaker'])}: {s['text']}" for s in segs
            )
        else:
            text = " ".join(s["text"] for s in segs)
        return {"id": len(chunks), "start": segs[0]["start"], "end": segs[-1]["end"], "text": text}

    for seg in segments:
        w = len(seg["text"].split())
        if cur and words + w > max_words:
            chunks.append(make_chunk(cur))
            cur, words = [], 0
        cur.append(seg)
        words += w
    if cur:
        chunks.append(make_chunk(cur))
    return chunks

qa_cache = {}

def get_job_index(job):
    job_id = job["id"]
    if job_id not in qa_cache:
        chunks = build_chunks(job["segments"], job["purpose"], job.get("speaker_names"))
        embs = retriever_model.encode(
            [c["text"] for c in chunks], convert_to_tensor=True, normalize_embeddings=True
        )
        qa_cache[job_id] = (chunks, embs)
    return qa_cache[job_id]

def retrieve(question, chunks, embeddings, top_k=6, candidates=20):
    q = retriever_model.encode(QUERY_PREFIX + question, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(q, embeddings)[0]
    idx = torch.topk(scores, min(candidates, len(chunks))).indices.tolist()
    rr = reranker.predict([(question, chunks[i]["text"]) for i in idx])
    best = sorted(zip(idx, rr), key=lambda x: -x[1])[:top_k]
    return [chunks[i] for i in sorted(i for i, _ in best)]

def mmss(t):
    t = int(t)
    return f"{t // 60}:{t % 60:02d}"

QA_PROMPT = """Answer the question using ONLY the transcript excerpts below.
Each excerpt has an id in square brackets and a time range.
After each claim, cite the excerpt ids it came from, like [2] or [2][5].
If the excerpts do not contain the answer, say so plainly and cite nothing.

Excerpts:
{context}

Question: {question}

Answer:"""

def call_gemini(prompt, max_retries=3):
    last_error = None
    for model in ["gemini-flash-latest", "gemini-3.6-flash"]:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model, contents=prompt,
                    config=types.GenerateContentConfig(
                        thinking_config=types.ThinkingConfig(thinking_level="low")
                    )
                )
                return response.text.strip()
            except Exception as e:
                last_error = e
                print(f"API call failed ({model}, attempt {attempt + 1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(min(60, 5 * (2 ** attempt)))
    raise last_error if last_error else RuntimeError("Gemini call failed")

BROAD_PATTERNS = [
    r"\bsummar(y|ize|ise)\b", r"\boverall\b", r"\bmain (idea|point|topic)s?\b",
    r"\bwhat (is|was) (this|the (meeting|talk|story|speech)) about\b",
    r"\bthemes?\b", r"\bconclusion\b", r"\bpurpose\b",
]

def is_broad_question(question):
    q = question.lower()
    return any(re.search(p, q) for p in BROAD_PATTERNS)

def answer_with_sources(question, job):
    chunks, embs = get_job_index(job)
    retrieved = retrieve(question, chunks, embs)

    context = "\n\n".join(
        f"[{c['id']}] ({mmss(c['start'])}-{mmss(c['end'])})\n{c['text']}" for c in retrieved
    )
    raw = call_gemini(QA_PROMPT.format(context=context, question=question))

    cited_ids = {int(n) for n in re.findall(r"\[(\d+)\]", raw)}
    answer = re.sub(r"\s*\[\d+\]", "", raw).strip()

    sources = [
        {"id": c["id"], "start": c["start"], "end": c["end"], "text": c["text"]}
        for c in retrieved if c["id"] in cited_ids
    ]
    return answer, sources

In [ ]:
from supabase import create_client
from google.colab import userdata
import requests

supabase_url = userdata.get("SUPABASE_URL")
supabase_key = userdata.get("SUPABASE_SERVICE_ROLE_KEY")
supabase_anon_key = userdata.get("SUPABASE_ANON_KEY")

supabase = create_client(supabase_url, supabase_key)

In [ ]:
try:
    supabase.storage.create_bucket("job-files", options={"public": True})
    print("Bucket created")
except Exception as e:
    print("Bucket may already exist:", e)

In [ ]:
import os
import subprocess

def save_upload_text(job_id: str, content: str, filename: str) ->str:
  local_path = f"/content/{job_id}_{filename}"

  with open(local_path, "w", encoding = "utf-8") as f:
    f.write(content)

  storage_path = f"{job_id}/{filename}"

  with open(local_path, "rb") as f:
    supabase.storage.from_("job-files").upload(
        storage_path,
        f,
        file_options = {"content-type": "text/plain", "upsert": "true"}
    )

  os.remove(local_path)
  return storage_path

def get_signed_url(storage_path: str, expires_in: int = 3600) -> str:
  if not storage_path:
    return None

  marker = "/job-files/"
  if marker in storage_path:
    storage_path = storage_path.split(marker, 1)[1]

  try:
    result = supabase.storage.from_("job-files").create_signed_url(storage_path, expires_in)
    return result.get("signedURL") or result.get("signed_url")

  except Exception as e:
    print(f"Failed to create signed URL for {storage_path}: {e}")
    return None

def make_playback_audio(input_path: str) -> str:
  out = os.path.splitext(input_path)[0] + "_play.mp3"
  try:
    subprocess.run(
      ["ffmpeg", "-y", "-i", input_path, "-vn", "-ac", "1", "-b:a", "64k", out],
      check = True,
      capture_output = True,
      text = True,
    )
  except subprocess.CalledProcessError as e:
    raise RuntimeError(f"ffmpeg failed: {e.stderr[-500:]}")
  return out

def save_upload_file(job_id: str, local_path: str, filename: str, content_type: str) -> str:
  storage_path = f"{job_id}/{filename}"
  with open(local_path, "rb") as f:
    supabase.storage.from_("job-files").upload(
        storage_path,
        f,
        file_options = {"content-type": content_type, "upsert": "true"}
    )

  return storage_path

In [ ]:
from fastapi import Header, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

security = HTTPBearer()

def get_current_user_id(credentials: HTTPAuthorizationCredentials = Depends(security)):

  token = credentials.credentials
  try:
    user_response = supabase.auth.get_user(token)
    return user_response.user.id
  except Exception:
    raise HTTPException(status_code=401, detail="Invalid or expired token")

print("Auth Helper ready")

In [ ]:
from fastapi import FastAPI, UploadFile, BackgroundTasks, Depends, Form, HTTPException
from pydantic import BaseModel
import shutil
import uuid
import os
from fastapi.middleware.cors import CORSMiddleware
import queue
import threading

ALLOWED_EXT = {".mp3", ".wav", ".m4a", ".mp4", ".webm", ".ogg", ".flac", ".aac", ".opus", ".3gp", ".wma"}
MAX_UPLOAD_BYTES = 200 * 1024 * 1024

job_queue = queue.Queue()

def worker_loop():
  while True:
    job_id, file_path, purpose = job_queue.get()
    try:
      process_audio(job_id, file_path, purpose)
    except Exception as e:
      print(f"worker crashed on job {job_id}: {e}")
    finally:
      job_queue.task_done()

supabase.table("jobs").update({"status": "Failed", "error": "Server restarted during processing"}).in_(
    "status", ["Queued", "Transcribing", "Diarizing", "Summarizing", "Formatting", "Generating files"]
).execute()

worker_thread = threading.Thread(target = worker_loop, daemon = True)
worker_thread.start()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def get_root():
  return {"message": "This is my backend."}

class SignupRequest(BaseModel):
  email: str
  password: str
  full_name: str
  date_of_birth: str
  invite_code: str

@app.post("/auth/signup")
def signup(request: SignupRequest):
  code_result = supabase.table("invite_codes").select("*").eq("code", request.invite_code).eq("active", True).execute()
  if not code_result.data:
    raise HTTPException(status_code = 400, detail = " Invalid Invide Code")

  code_row = code_result.data[0]
  if code_row["used_count"] >= code_row["max_uses"]:
    raise HTTPException(status_code = 400, detail = "This Invite Code has already been used.")

  try:
    supabase.auth.admin.create_user({
        "email": request.email,
        "password": request.password,
        "email_confirm": True,
        "user_metadata": {
            "full_name": request.full_name,
            "date_of_birth": request.date_of_birth,
        }
    })

  except Exception as e:
    raise HTTPException(status_code = 400, detail = f"Signup failed: {e}")

  supabase.table("invite_codes").update({"used_count": code_row["used_count"] + 1}).eq("id", code_row["id"]).execute()

  try:
    resp = requests.post(
        f"{supabase_url}/auth/v1/token?grant_type=password",
        headers={
            "apikey": supabase_anon_key,
            "Content-Type": "application/json",
        },
        json={"email": request.email, "password": request.password},
    )
    resp.raise_for_status()
    session_data = resp.json()

  except Exception as e:
    raise HTTPException(status_code = 400, detail = f"Sign in failed: {e}")

  return {
      "access_token": session_data["access_token"],
      "refresh_token": session_data["refresh_token"],
      "user": {
          "id": session_data["user"]["id"],
          "email": session_data["user"]["email"],
          "user_metadata": session_data["user"].get("user_metadata", {}),
      }
  }

def process_audio(job_id: str, file_path: str, purpose:str):
  try:
    supabase.table("jobs").update({"status": "Transcribing"}).eq("id", job_id).execute()

    segments_generator, info = whisper_model.transcribe(file_path, language = "en", vad_filter = True, vad_parameters = dict(min_silence_duration_ms = 500), condition_on_previous_text = False, beam_size = 5, word_timestamps=True)
    whisper_segments = list(segments_generator)
    transcript = " ".join(seg.text.strip() for seg in whisper_segments)

    LOW_CONF_THRESHOLD = 0.75
    transcript_word_confidence = []
    for seg in whisper_segments:
      if seg.words:
        for w in seg.words:
          if w.probability < LOW_CONF_THRESHOLD:
            transcript_word_confidence.append({
                    "word": w.word.strip(),
                    "start": round(w.start, 2),
                    "end": round(w.end, 2),
                    "confidence": round(w.probability, 3),
                })

    supabase.table("jobs").update({"status": "Diarizing"}).eq("id", job_id).execute()

    speaker_segments = diarize_audio(file_path)
    labelled_segments = assign_speakers_to_transcript(whisper_segments, speaker_segments, LOW_CONF_THRESHOLD)
    speaker_transcript = format_speaker_transcription(labelled_segments)

    supabase.table("jobs").update({"status": "Summarizing"}).eq("id", job_id).execute()

    summary_input = speaker_transcript if purpose == "meeting" else transcript
    summary = summarize_transcript(summary_input, purpose = purpose)

    supabase.table("jobs").update({"status": "Formatting"}).eq("id", job_id).execute()
    formatted_output = generate_formatted_output(purpose, summary_input, summary)

    supabase.table("jobs").update({"status": "Generating files"}).eq("id", job_id).execute()

    transcript_content = speaker_transcript if purpose == "meeting" else transcript
    transcript_url = save_upload_text(job_id, transcript_content, "transcript.txt")
    summary_url = save_upload_text(job_id, summary, "summary.txt")

    audio_path = None
    try:
      play_path = make_playback_audio(file_path)
      try:
        audio_path = save_upload_file(job_id, play_path, "audio.mp3", "audio/mpeg")
      finally:
        if os.path.exists(play_path):
          os.remove(play_path)

    except Exception as e:
      print(f"Playback audio failed for {job_id}: {e}")

    supabase.table("jobs").update({
        "status": "Done",
        "transcript": transcript,
        "speaker_transcript": speaker_transcript,
        "summary": summary,
        "transcript_url": transcript_url,
        "summary_url": summary_url,
        "transcript_word_confidence": transcript_word_confidence,
        "formatted_output": formatted_output,
        "segments": labelled_segments,
        "audio_url": audio_path,
    }).eq("id", job_id).execute()

  except Exception as e:
    supabase.table("jobs").update({"status": "Failed", "error": str(e)}).eq("id",job_id).execute()
    print(f"Job {job_id} failed: {e}")

  finally:
    if os.path.exists(file_path):
      os.remove(file_path)

@app.post("/upload")
def upload_file(file: UploadFile, purpose: str = "meeting", title: str = Form(None), user_id: str = Depends(get_current_user_id)):
  if purpose not in SUMMARY_PROMPTS:
    raise HTTPException(status_code = 400, detail = "Invalid purpose.")

  ext = os.path.splitext(file.filename or "")[1].lower()
  if ext not in ALLOWED_EXT:
    raise HTTPException(status_code = 400, detail = "Unsupported file type.")

  job_id = str(uuid.uuid4())
  file_path = f"/content/{job_id}{ext}"

  size = 0
  with open(file_path, "wb") as f:
    while chunk := file.file.read(1024 * 1024):
      size += len(chunk)
      if size > MAX_UPLOAD_BYTES:
        f.close()
        os.remove(file_path)
        raise HTTPException(status_code = 413, detail = "File too large.")
      f.write(chunk)

  supabase.table("jobs").insert({"id": job_id, "user_id": user_id, "status": "Queued", "purpose": purpose, "filename": file.filename, "title": title}).execute()

  job_queue.put((job_id, file_path, purpose))

  return {"job_id":job_id, "status":"Queued", "queue_position": job_queue.qsize()}

@app.get("/status/{job_id}")
def check_status(job_id: str, user_id:str = Depends(get_current_user_id)):
  result = supabase.table("jobs").select("*").eq("id", job_id).eq("user_id", user_id).execute()
  if not result.data:
    raise HTTPException(status_code = 404, detail = "Job not found.")

  return enrich_job(result.data[0])

class TitleUpdate(BaseModel):
  title: str

@app.patch("/jobs/{job_id}/title")
def update_job_title(job_id: str, request: TitleUpdate, user_id: str = Depends(get_current_user_id)):
  result = supabase.table("jobs").update({"title": request.title}).eq("id", job_id).eq("user_id", user_id).execute()
  if not result.data:
    raise HTTPException(status_code = 404, detail = "Job not found.")

  return enrich_job(result.data[0])

@app.get("/my-jobs")
def get_my_jobs(user_id:str = Depends(get_current_user_id)):
  result = supabase.table("jobs").select(
      "id,status,purpose,filename,title,summary,transcript_url,summary_url,created_at,error"
  ).eq("user_id", user_id).order("created_at", desc = True).execute()

  jobs = result.data

  for job in jobs:
    job["transcript_url"] = get_signed_url(job.get("transcript_url"))
    job["summary_url"] = get_signed_url(job.get("summary_url"))

  return jobs

@app.get("/jobs/{job_id}/messages")
def get_job_messages(job_id: str, user_id: str = Depends(get_current_user_id)):
  job_result = supabase.table("jobs").select("*").eq("id", job_id).eq("user_id", user_id).execute()
  if not job_result.data:
    raise HTTPException(status_code = 404, detail = "Job not found.")
  job = enrich_job(job_result.data[0])

  messages_result = supabase.table("qa_messages").select("*").eq("job_id", job_id).order("created_at", desc = False).execute()

  return {
        "job_id": job_id,
        "filename": job["filename"],
        "purpose": job["purpose"],
        "status": job["status"],
        "transcript_url": job["transcript_url"],
        "summary_url": job["summary_url"],
        "audio_url": job["audio_url"],
        "summary": job.get("summary"),
        "segments": job.get("segments"),
        "speaker_names": job.get("speaker_names") or {},
        "display_transcript": job.get("display_transcript"),
        "messages": messages_result.data
    }

class QuestionRequest(BaseModel):
    question: str

@app.post("/ask/{job_id}")
def ask_question(job_id: str, request: QuestionRequest, user_id: str = Depends(get_current_user_id)):
    result = supabase.table("jobs").select("*").eq("id", job_id).eq("user_id", user_id).execute()
    if not result.data:
        raise HTTPException(status_code=404, detail="Job not found.")
    job = result.data[0]
    if job["status"] != "Done":
        raise HTTPException(status_code=409, detail="Job not finished processing yet.")

    qa_source = job["speaker_transcript"] if job["purpose"] == "meeting" else job["transcript"]

    if not job.get("segments") or is_broad_question(request.question):
        answer = answer_thematic_questions(qa_source, request.question)
        sources = []
    else:
        answer, sources = answer_with_sources(request.question, job)

    supabase.table("qa_messages").insert({
        "job_id": job_id, "question": request.question,
        "answer": answer, "sources": sources
    }).execute()
    return {"question": request.question, "answer": answer, "sources": sources}

class SpeakerRename(BaseModel):
  speaker_names: dict[str, str]

def render_speaker_transcript(segments, names):
  lines, current, buf = [], None, []
  for s in segments:
    if s["speaker"] != current:
      if buf:
        lines.append(f"{names.get(current, current)}: {' '.join(buf)}")
      current, buf = s["speaker"], [s["text"]]

    else:
      buf.append(s["text"])

  if buf:
    lines.append(f"{names.get(current, current)}: {' '.join(buf)}")
  return "\n".join(lines)

def enrich_job(job):
  job["transcript_url"] = get_signed_url(job.get("transcript_url"))
  job["summary_url"] = get_signed_url(job.get("summary_url"))
  job["audio_url"] = get_signed_url(job.get("audio_url"), expires_in = 6 * 3600)

  if job.get("segments"):
    job["display_transcript"] = render_speaker_transcript(job["segments"], job.get("speaker_names") or {})

  return job

@app.patch("/jobs/{job_id}/speakers")
def rename_speakers(job_id: str, request: SpeakerRename, user_id: str = Depends(get_current_user_id)):
  cleaned = {k: v.strip()[:50] for k, v in request.speaker_names.items() if v.strip()}
  result = supabase.table("jobs").update({"speaker_names": cleaned}).eq("id", job_id).eq("user_id", user_id).execute()

  if not result.data:
    raise HTTPException(status_code = 404, detail = "Job not found.")

  qa_cache.pop(job_id, None)
  return enrich_job(result.data[0])

@app.delete("/jobs/{job_id}")
def delete_job(job_id: str, user_id: str = Depends(get_current_user_id)):
  result = supabase.table("jobs").select("id").eq("id", job_id).eq("user_id", user_id).execute()
  if not result.data:
    raise HTTPException(status_code = 404, detail = "Job not found.")

  supabase.table("qa_messages").delete().eq("job_id", job_id).execute()

  try:
    supabase.storage.from_("job-files").remove([
        f"{job_id}/transcript.txt",
        f"{job_id}/summary.txt",
        f"{job_id}/audio.mp3"
    ])
  except Exception as e:
    print(f"Storage cleanup warning for job {job_id}: {e}")

  supabase.table("jobs").delete().eq("id", job_id).eq("user_id", user_id).execute()

  qa_cache.pop(job_id, None)

  return {"job_id": job_id, "deleted": True}


In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok_token = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000)
print("Your public URL:", public_url)

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()